# Polaris Privilege Hierarchy — Cascade & Inheritance Test

Empirically verifies the two dimensions Apache Polaris 1.3.0 uses to govern access
(`privilege/doc-privilege-test.md` §1) instead of assuming them:

1. **Privilege power cascade** — a coarse master encompasses fine-grained children:
   ```
   CATALOG_MANAGE_CONTENT  ⊇  NAMESPACE_CREATE/DROP · TABLE_CREATE/DROP · VIEW_CREATE/DROP
   CATALOG_MANAGE_METADATA ⊇  TABLE_LIST/READ/WRITE · VIEW_LIST
   ```
   We grant **only** the master to a worker and check every action: in-branch children
   must come back `AUTHORIZED`, out-of-branch actions `BLOCKED_PRIV`.
2. **Securable inheritance** — a catalog-scoped grant cascades *down* the resource tree
   (catalog → namespace → table/view). We grant at the catalog-role level and confirm the
   worker can act on a **nested** entity.

Built on the Phase-0 scaff/worker lifecycle (`case_scaffold`, `instance_principal`). Root
only bootstraps/destroys; every challenge runs as a single-use worker. Same six-way
taxonomy as `polaris_privilege_matrix_test.ipynb` — `BLOCKED_OPA`/`BLOCKED_CONFIG` should
never appear; if they do they flag a harness bug, not a real boundary.

> **Expectations = the measured baseline** (`doc-privilege-results.md` §3, 2026-07-02): on
> this build both coarse masters authorize *every* action, so the expected cascade is "all
> AUTHORIZED" for both. The notebook now verifies the build still matches that recorded
> baseline — a ❗/DEVIATION marks a **regression**, not a disagreement with the source doc.

**Case structure** — every probe is one test case composed by `run_case` (in `src`):
**Suite** (`build_suite`, common) arranges a fresh catalog + scaffold + worker and *records*
every entity it creates → **Execute** (per-action) fires the challenge → **Result** (`classify`,
pure) derives the outcome → **Cleaner** (`clean`, common) deletes the recorded entities in a
`finally`. Cleanup targets tracked entities, not a naming guess, so no case can leak into another.

In [1]:
# ── src path bootstrap (shared modules live in /src) ────────────────────────────
import sys, pathlib
_ROOT = pathlib.Path.cwd()
while not (_ROOT / "src").is_dir() and _ROOT != _ROOT.parent:
    _ROOT = _ROOT.parent
_SRC = _ROOT / "src"
if str(_SRC) not in sys.path:
    sys.path.insert(0, str(_SRC))

import time, uuid
import importlib, nb_support
importlib.reload(nb_support)
from nb_support import *
init_env("local")                       # default safe target; switch to "dev" for shared cluster
require_not_prod("privilege hierarchy / role mutations")  # hard-fail against company PROD

# ── global request timeout: no single HTTP call may hang the notebook ────────
# Patches the shared requests.Session so EVERY call (notebook + src utils) gets a
# (connect, read) timeout. A dead/slow endpoint now raises promptly instead of
# blocking forever — probe() catches it and records ERROR.
import requests as _rq
if not getattr(_rq.sessions.Session, "_timeout_patched", False):
    _orig_request = _rq.sessions.Session.request
    def _request_with_timeout(self, method, url, **kw):
        kw.setdefault("timeout", (5, 30))          # 5s connect, 30s read
        return _orig_request(self, method, url, **kw)
    _rq.sessions.Session.request = _request_with_timeout
    _rq.sessions.Session._timeout_patched = True

tok = root_token()
_RAW_SETTLE = 0.6                        # settle after a read-after-write lag-500
NS = lambda cat: f"ns-{cat}"            # per-catalog namespace (Vector B: no flat strings)
print("✅ privilege harness loaded")

🟢 LOCAL   POLARIS_ENV=local   (Polaris 1.3.0)
   Polaris: http://192.168.139.2:8181
   MinIO:   http://192.168.139.2:9000  bucket=data-catalog-bucket
   OpenSearch: localhost:9200  index=k8s-logs-*
   purge_deletes_files=False
🟢 LOCAL   POLARIS_ENV=local   (Polaris 1.3.0)
   Polaris: http://192.168.139.2:8181
   MinIO:   http://192.168.139.2:9000  bucket=data-catalog-bucket
   OpenSearch: localhost:9200  index=k8s-logs-*
   purge_deletes_files=False
🟢 LOCAL   POLARIS_ENV=local   (Polaris 1.3.0)
   Polaris: http://192.168.139.2:8181
   MinIO:   http://192.168.139.2:9000  bucket=data-catalog-bucket
   OpenSearch: localhost:9200  index=k8s-logs-*
   purge_deletes_files=False
✅ privilege harness loaded


In [2]:
# ── action registry + entity bodies ──────────────────────────────────────────
# Each action: build(cat,hdr,ctx) scaffolds the prereq AS THE SCAFF principal;
# challenge(cat,hdr,ctx) fires the action under test AS THE WORKER.
def _tbl_body(cat, ns, tbl):
    return {"name": tbl, "location": f"s3a://{BUCKET}/{cat}/{ns}/{tbl}/",
            "schema": {"type":"struct","fields":[
                {"id":1,"name":"id","type":"long","required":True},
                {"id":2,"name":"value","type":"string","required":False}]}}

def _view_body(cat, ns, vw):
    return {"name": vw, "default-namespace":[ns],
            "schema":{"type":"struct","schema-id":0,
                      "fields":[{"id":1,"name":"id","type":"long","required":True}]},
            "view-version":{"version-id":1,"timestamp-ms":int(time.time()*1000),
                            "schema-id":0,"default-namespace":[ns],"summary":{"engine":"spark"},
                            "representations":[{"type":"sql","sql":"SELECT 1 AS id","dialect":"spark"}]},
            "location": f"s3a://{BUCKET}/{cat}/{ns}/{vw}/"}

def _mk_ns(cat, hdr):
    requests.post(f"{BASE_CAT}/{cat}/namespaces", headers=hdr,
                  json={"namespace":[NS(cat)], "properties":{}})
def _mk_tbl(cat, hdr, ctx):
    requests.post(f"{BASE_CAT}/{cat}/namespaces/{NS(cat)}/tables", headers=hdr,
                  json=_tbl_body(cat, NS(cat), ctx["entity"]))
def _mk_view(cat, hdr, ctx):
    requests.post(f"{BASE_CAT}/{cat}/namespaces/{NS(cat)}/views", headers=hdr,
                  json=_view_body(cat, NS(cat), ctx["entity"]))

# action_id -> spec.  drop=True actions get the config gate neutralized (gap 1).
ACTIONS = {
  "ns_create":  dict(target="Namespace", action="CREATE", drop=False,
     build=lambda cat,hdr,ctx: None,
     challenge=lambda cat,hdr,ctx: requests.post(
         f"{BASE_CAT}/{cat}/namespaces", headers=hdr,
         json={"namespace":[NS(cat)], "properties":{}})),
  "ns_list":    dict(target="Namespace", action="LIST", drop=False,
     build=lambda cat,hdr,ctx: _mk_ns(cat,hdr),
     challenge=lambda cat,hdr,ctx: requests.get(
         f"{BASE_CAT}/{cat}/namespaces", headers=hdr)),
  "ns_drop":    dict(target="Namespace", action="DROP", drop=True,
     build=lambda cat,hdr,ctx: _mk_ns(cat,hdr),
     challenge=lambda cat,hdr,ctx: requests.delete(
         f"{BASE_CAT}/{cat}/namespaces/{NS(cat)}", headers=hdr)),
  "tbl_create": dict(target="Table", action="CREATE", drop=False,
     build=lambda cat,hdr,ctx: _mk_ns(cat,hdr),
     challenge=lambda cat,hdr,ctx: requests.post(
         f"{BASE_CAT}/{cat}/namespaces/{NS(cat)}/tables", headers=hdr,
         json=_tbl_body(cat, NS(cat), ctx["entity"]))),
  "tbl_list":   dict(target="Table", action="LIST", drop=False,
     build=lambda cat,hdr,ctx: (_mk_ns(cat,hdr), _mk_tbl(cat,hdr,ctx)),
     challenge=lambda cat,hdr,ctx: requests.get(
         f"{BASE_CAT}/{cat}/namespaces/{NS(cat)}/tables", headers=hdr)),
  "tbl_read":   dict(target="Table", action="READ", drop=False,
     build=lambda cat,hdr,ctx: (_mk_ns(cat,hdr), _mk_tbl(cat,hdr,ctx)),
     challenge=lambda cat,hdr,ctx: requests.get(
         f"{BASE_CAT}/{cat}/namespaces/{NS(cat)}/tables/{ctx['entity']}", headers=hdr)),
  "tbl_commit": dict(target="Table", action="COMMIT", drop=False,
     build=lambda cat,hdr,ctx: (_mk_ns(cat,hdr), _mk_tbl(cat,hdr,ctx)),
     challenge=lambda cat,hdr,ctx: requests.post(
         f"{BASE_CAT}/{cat}/namespaces/{NS(cat)}/tables/{ctx['entity']}", headers=hdr,
         json={"requirements":[], "updates":[
             {"action":"set-properties","updates":{"probe":"1"}}]})),
  "tbl_drop":   dict(target="Table", action="DROP", drop=True,
     build=lambda cat,hdr,ctx: (_mk_ns(cat,hdr), _mk_tbl(cat,hdr,ctx)),
     challenge=lambda cat,hdr,ctx: requests.delete(
         f"{BASE_CAT}/{cat}/namespaces/{NS(cat)}/tables/{ctx['entity']}", headers=hdr)),
  "view_create":dict(target="View", action="CREATE", drop=False,
     build=lambda cat,hdr,ctx: _mk_ns(cat,hdr),
     challenge=lambda cat,hdr,ctx: requests.post(
         f"{BASE_CAT}/{cat}/namespaces/{NS(cat)}/views", headers=hdr,
         json=_view_body(cat, NS(cat), ctx["entity"]))),
  "view_get":   dict(target="View", action="GET", drop=False,
     build=lambda cat,hdr,ctx: (_mk_ns(cat,hdr), _mk_view(cat,hdr,ctx)),
     challenge=lambda cat,hdr,ctx: requests.get(
         f"{BASE_CAT}/{cat}/namespaces/{NS(cat)}/views/{ctx['entity']}", headers=hdr)),
  "view_drop":  dict(target="View", action="DROP", drop=True,
     build=lambda cat,hdr,ctx: (_mk_ns(cat,hdr), _mk_view(cat,hdr,ctx)),
     challenge=lambda cat,hdr,ctx: requests.delete(
         f"{BASE_CAT}/{cat}/namespaces/{NS(cat)}/views/{ctx['entity']}", headers=hdr)),
}
LABEL = {k: f"{v['target']}.{v['action']}" for k,v in ACTIONS.items()}

# ── six-way classifier (identical taxonomy to the matrix notebook) ───────────
def _is_lag(r):
    b = r.text.lower()
    return r.status_code == 500 and ("metadata" in b or "nullpointer" in b)

def classify(r):
    sc, b = r.status_code, r.text.lower()
    if sc in (200, 201, 204):                        return "AUTHORIZED"
    if sc == 409 or "already exists" in b:           return "AUTHORIZED"   # create lag-committed
    if _is_lag(r):                                   return "LAG_500"
    if sc == 403:
        if "unable to purge" in b:                   return "BLOCKED_CONFIG"  # harness bug
        if "is not authorized" in b or "not authorized for op" in b: return "BLOCKED_PRIV"
        if "opa" in b or "ext_authz" in b or "denied by policy" in b: return "BLOCKED_OPA"  # harness bug
        return "BLOCKED_PRIV"
    if sc == 404:                                    return "NOT_FOUND_404"   # setup issue
    return f"HTTP_{sc}"

# ── one probe = one test case, composed via run_case (from src) ──────────────
# The four responsibilities (arrange / act / assert / cleanup):
#   Suite   = build_suite(...)   COMMON  — fresh catalog + scaff prereq + worker
#   Execute = spec["challenge"]  PER-CELL — fire the action under test
#   Result  = classify(...)      PURE    — derive the outcome, change nothing
#   Cleaner = clean(...)         COMMON  — reclaim every tracked entity, ALWAYS
# run_case wires them with a try/finally so cleanup is guaranteed, and cleanup
# deletes the *recorded* Entities (not a naming guess) — no probe can leak.
def probe(action_id, privileges, verbose=False):
    """Returns (outcome, granted, invalid). All resources are reclaimed before it
    returns — even on exception — because run_case cleans in a `finally`."""
    spec = ACTIONS[action_id]
    privs = list(privileges) if isinstance(privileges, (list, tuple)) else [privileges]

    def suite(ents, root):                               # 1. arrange (common)
        build_suite(ents, root, prefix=f"hzr-{action_id.replace('_','')}-",
                    build=lambda cat, hdr, ctx: spec["build"](cat, hdr, ctx),
                    footprint=privs, drop_gate=spec["drop"])

    def execute(suite_ents, created, root):              # 2. act (per-cell)
        c = suite_ents.ctx
        if c.get("worker_hdr") is None:
            return None                                  # token not issued
        fire = lambda: spec["challenge"](c["catalog"], c["worker_hdr"], c)
        r = fire()
        if _is_lag(r):                                   # read-after-write 500
            time.sleep(_RAW_SETTLE); r = fire()
        if (r.status_code == 404 and not spec["drop"]
                and spec["action"] in ("READ", "GET", "LIST")):   # entity-visibility lag
            time.sleep(_RAW_SETTLE); r = fire()
        if spec["action"] == "CREATE" and r.status_code < 400:
            created.ctx["made"] = c["entity"]            # record what the act created
        return r

    def result(name, resp, suite_ents, created):         # 3. assert (pure)
        c = suite_ents.ctx
        if resp is None:
            return "BLOCKED_OPA", c.get("granted", []), c.get("invalid", [])
        return classify(resp), c.get("granted", []), c.get("invalid", [])

    try:
        outcome, granted, invalid = run_case(action_id, suite, execute, result, root=tok)
        detail = ""
    except Exception as e:
        outcome, granted, invalid, detail = "ERROR", [], [], f"{type(e).__name__}: {str(e)[:110]}"
    if verbose or outcome in ("ERROR", "BLOCKED_OPA", "BLOCKED_CONFIG"):
        print(f"     ↳ {LABEL[action_id]}/{','.join(privs)} → {outcome}: {detail}")
    return outcome, granted, invalid

print(f"✅ {len(ACTIONS)} actions registered; run_case orchestrator ready "
      f"(Suite=build_suite · Execute=challenge · Result=classify · Cleaner=clean)")

✅ 11 actions registered; run_case orchestrator ready (Suite=build_suite · Execute=challenge · Result=classify · Cleaner=clean)


In [3]:
# ── EXPECTED CASCADE = MEASURED baseline (doc-privilege-results.md §3, 2026-07-02) ─
# Live runs showed BOTH coarse masters authorize EVERY action on this build
# (CATALOG_MANAGE_METADATA is NOT read-only). So the expected in-branch set for each
# master is ALL actions. The notebook now verifies the build still matches this
# recorded baseline — a ❗/DEVIATION flags a regression, not a doc disagreement.
ALL_ACTIONS = list(ACTIONS.keys())
CASCADE_CLAIM = {
  "CATALOG_MANAGE_CONTENT":  list(ALL_ACTIONS),
  "CATALOG_MANAGE_METADATA": list(ALL_ACTIONS),
}
print("Masters under test:", list(CASCADE_CLAIM))
print("Actions probed per master:", len(ALL_ACTIONS))

Masters under test: ['CATALOG_MANAGE_CONTENT', 'CATALOG_MANAGE_METADATA']
Actions probed per master: 11


In [4]:
# ── BLOCK A: coarse-master cascade ───────────────────────────────────────────
# Grant ONLY the master; run every action; compare measured outcome to the claim.
CASCADE = {}   # (master, action_id) -> outcome
print("Verifying privilege cascade (grant master alone, probe every action)...")
print("(each probe = fresh catalog + scaffold + worker + teardown; full run ~a few min)\n")
_t0 = time.time()
for master, in_branch in CASCADE_CLAIM.items():
    print(f"━━ {master} ━━")
    for aid in ALL_ACTIONS:
        outcome, _g, inv = probe(aid, [master])
        CASCADE[(master, aid)] = outcome
        expected = "AUTHORIZED" if aid in in_branch else "BLOCKED_PRIV"
        match = "✓" if outcome == expected else "✗ MISMATCH"
        note = f"  (∅ invalid: {inv})" if inv else ""
        print(f"  {LABEL[aid]:16} → {outcome:14} expect {expected:13} {match}{note}  "
              f"[{time.time()-_t0:5.0f}s]")
    print()
print(f"✅ cascade probed: {len(CASCADE)} (master×action) cells in {time.time()-_t0:.0f}s")

Verifying privilege cascade (grant master alone, probe every action)...
(each probe = fresh catalog + scaffold + worker + teardown; full run ~a few min)

━━ CATALOG_MANAGE_CONTENT ━━
  ⚠️ grant_privilege CATALOG_MANAGE_CONTENT on hzr-nscreate-1ea676/hzr-nscreate-1ea676-scaff-crole → [404] {"error":{"message":"Catalog not found: hzr-nscreate-1ea676","type":"NotFoundException","code":404}}
  ⚠️ grant_privilege CATALOG_MANAGE_CONTENT on hzr-nscreate-1ea676/hzr-nscreate-1ea676-worker-crole → [404] {"error":{"message":"Catalog not found: hzr-nscreate-1ea676","type":"NotFoundException","code":404}}
  Namespace.CREATE → NOT_FOUND_404  expect AUTHORIZED    ✗ MISMATCH  (∅ invalid: ['CATALOG_MANAGE_CONTENT'])  [   23s]


KeyboardInterrupt: 

In [33]:
# ── render the cascade truth table ───────────────────────────────────────────
# Plain-text table first (always visible), rich Markdown as a bonus.
SYM = {"AUTHORIZED":"✅","BLOCKED_PRIV":"🔒","GRANT_INVALID":"∅","BLOCKED_OPA":"⚠️OPA",
       "BLOCKED_CONFIG":"⚠️CFG","LAG_500":"…","NOT_FOUND_404":"404","ERROR":"💥"}
masters = list(CASCADE_CLAIM)
# 1) guaranteed-visible plain-text table -------------------------------------
w = 16
print("POLARIS 1.3.0 — PRIVILEGE CASCADE TRUTH TABLE\n")
print(f"{'Action':{w}} " + " ".join(f"{m.replace('CATALOG_MANAGE_','CM_'):16}" for m in masters) + " claimed_under")
print("-" * (w + 1 + 17*len(masters) + 14))
for aid in ALL_ACTIONS:
    cells = []
    for m in masters:
        out = CASCADE.get((m, aid)); exp_auth = aid in CASCADE_CLAIM[m]
        flag = "!" if (out and (out == "AUTHORIZED") != exp_auth) else " "
        cells.append(f"{(out or '·')+flag:16}")
    claimed = ",".join(m.replace("CATALOG_MANAGE_","CM_") for m in masters if aid in CASCADE_CLAIM[m]) or "—"
    print(f"{LABEL[aid]:{w}} " + " ".join(cells) + f" {claimed}")
print("\nLegend: '!' after a cell = DEVIATES from the recorded 2026-07-02 baseline "
      "(a regression to investigate). ⚠️ BLOCKED_OPA/BLOCKED_CONFIG = harness bug.")

# 2) bonus: rich Markdown render ---------------------------------------------
try:
    from IPython.display import Markdown, display
    hdr = "| Action | " + " | ".join(masters) + " | claimed under |"
    sep = "|" + "---|"*(len(masters)+2)
    lines = []
    for aid in ALL_ACTIONS:
        cs = []
        for m in masters:
            out = CASCADE.get((m, aid)); exp_auth = aid in CASCADE_CLAIM[m]
            s = SYM.get(out, out or "·")
            if out and ((out == "AUTHORIZED") != exp_auth): s += "❗"
            cs.append(s)
        claimed = ", ".join(m.replace("CATALOG_MANAGE_","CM_") for m in masters if aid in CASCADE_CLAIM[m]) or "—"
        lines.append(f"| {LABEL[aid]} | " + " | ".join(cs) + f" | {claimed} |")
    legend = ("\n**Legend:** ✅ AUTHORIZED · 🔒 BLOCKED_PRIV · ∅ GRANT_INVALID · ❗ deviates "
              "from the recorded 2026-07-02 baseline (regression) · ⚠️ harness bug")
    display(Markdown(f"### Polaris 1.3.0 — Privilege Cascade Truth Table\n\n{hdr}\n{sep}\n"
                     + "\n".join(lines) + legend))
except Exception:
    pass

POLARIS 1.3.0 — PRIVILEGE CASCADE TRUTH TABLE

Action           CM_CONTENT       CM_METADATA      claimed_under
-----------------------------------------------------------------
Namespace.CREATE AUTHORIZED       AUTHORIZED       CM_CONTENT,CM_METADATA
Namespace.LIST   AUTHORIZED       AUTHORIZED       CM_CONTENT,CM_METADATA
Namespace.DROP   AUTHORIZED       AUTHORIZED       CM_CONTENT,CM_METADATA
Table.CREATE     AUTHORIZED       AUTHORIZED       CM_CONTENT,CM_METADATA
Table.LIST       AUTHORIZED       AUTHORIZED       CM_CONTENT,CM_METADATA
Table.READ       AUTHORIZED       AUTHORIZED       CM_CONTENT,CM_METADATA
Table.COMMIT     AUTHORIZED       AUTHORIZED       CM_CONTENT,CM_METADATA
Table.DROP       AUTHORIZED       AUTHORIZED       CM_CONTENT,CM_METADATA
View.CREATE      AUTHORIZED       AUTHORIZED       CM_CONTENT,CM_METADATA
View.GET         AUTHORIZED       AUTHORIZED       CM_CONTENT,CM_METADATA
View.DROP        AUTHORIZED       AUTHORIZED       CM_CONTENT,CM_METADATA

Legend:

### Polaris 1.3.0 — Privilege Cascade Truth Table

| Action | CATALOG_MANAGE_CONTENT | CATALOG_MANAGE_METADATA | claimed under |
|---|---|---|---|
| Namespace.CREATE | ✅ | ✅ | CM_CONTENT, CM_METADATA |
| Namespace.LIST | ✅ | ✅ | CM_CONTENT, CM_METADATA |
| Namespace.DROP | ✅ | ✅ | CM_CONTENT, CM_METADATA |
| Table.CREATE | ✅ | ✅ | CM_CONTENT, CM_METADATA |
| Table.LIST | ✅ | ✅ | CM_CONTENT, CM_METADATA |
| Table.READ | ✅ | ✅ | CM_CONTENT, CM_METADATA |
| Table.COMMIT | ✅ | ✅ | CM_CONTENT, CM_METADATA |
| Table.DROP | ✅ | ✅ | CM_CONTENT, CM_METADATA |
| View.CREATE | ✅ | ✅ | CM_CONTENT, CM_METADATA |
| View.GET | ✅ | ✅ | CM_CONTENT, CM_METADATA |
| View.DROP | ✅ | ✅ | CM_CONTENT, CM_METADATA |
**Legend:** ✅ AUTHORIZED · 🔒 BLOCKED_PRIV · ∅ GRANT_INVALID · ❗ deviates from the recorded 2026-07-02 baseline (regression) · ⚠️ harness bug

In [34]:
# ── BLOCK B: securable inheritance (catalog grant → nested entity) ───────────
# Grant CATALOG_MANAGE_CONTENT at the catalog-role level, build a NESTED table and
# view under the namespace, and confirm the worker can act on the deep entity.
# AUTHORIZED proves the grant cascaded DOWN the securable tree (catalog→ns→entity).
print("Securable inheritance: catalog-scoped grant must reach nested entities...\n")
inherit_results = {}
for aid in ["tbl_read", "tbl_drop", "view_get", "view_drop"]:
    outcome, _g, _inv = probe(aid, ["CATALOG_MANAGE_CONTENT"])
    inherit_results[aid] = outcome
    ok = "✅ inherited" if outcome == "AUTHORIZED" else f"⚠️ {outcome}"
    print(f"  catalog-grant → {LABEL[aid]:16} on nested entity → {outcome:14} {ok}")
cascaded = all(o == "AUTHORIZED" for o in inherit_results.values())
print(f"\n{'✅' if cascaded else '⚠️'} securable inheritance "
      f"{'CONFIRMED' if cascaded else 'INCOMPLETE'}: a catalog-level grant "
      f"{'reaches' if cascaded else 'did NOT reach'} entities nested below it.")

Securable inheritance: catalog-scoped grant must reach nested entities...

  catalog-grant → Table.READ       on nested entity → AUTHORIZED     ✅ inherited
  catalog-grant → Table.DROP       on nested entity → AUTHORIZED     ✅ inherited
  catalog-grant → View.GET         on nested entity → AUTHORIZED     ✅ inherited
  catalog-grant → View.DROP        on nested entity → AUTHORIZED     ✅ inherited

✅ securable inheritance CONFIRMED: a catalog-level grant reaches entities nested below it.


In [35]:
# ── SUMMARY: match vs deviation against the recorded baseline ────────────────
print("CASCADE VERDICT (measured vs recorded 2026-07-02 baseline):\n")
confirmed, refuted = [], []
for master, in_branch in CASCADE_CLAIM.items():
    for aid in ALL_ACTIONS:
        out = CASCADE.get((master, aid))
        exp_auth = aid in in_branch
        if out in ("AUTHORIZED", "BLOCKED_PRIV"):
            (confirmed if (out == "AUTHORIZED") == exp_auth else refuted).append(
                (master, LABEL[aid], out, "AUTHORIZED" if exp_auth else "BLOCKED_PRIV"))
print(f"  ✅ edges MATCH baseline    : {len(confirmed)}")
print(f"  ❗ edges DEVIATE (regress) : {len(refuted)}")
for m, lab, got, exp in refuted:
    print(f"       {m} / {lab}: measured {got}, baseline expected {exp}")

flags = [(m,a,o) for (m,a),o in CASCADE.items()
         if o in ("BLOCKED_OPA","BLOCKED_CONFIG","LAG_500","ERROR")]
print("\nHARNESS-BUG / INCONCLUSIVE FLAGS (should be empty in a clean run):")
if flags:
    for m,a,o in flags: print(f"  ⚠️ {m} / {LABEL[a]} → {o}")
else:
    print("  ✅ none — every cascade cell is a clean AUTHORIZED / BLOCKED_PRIV")
print("\nNote: per-action GRANT_INVALID names (if any) were printed inline in Block A.")

CASCADE VERDICT (measured vs recorded 2026-07-02 baseline):

  ✅ edges MATCH baseline    : 22
  ❗ edges DEVIATE (regress) : 0

HARNESS-BUG / INCONCLUSIVE FLAGS (should be empty in a clean run):
  ✅ none — every cascade cell is a clean AUTHORIZED / BLOCKED_PRIV

Note: per-action GRANT_INVALID names (if any) were printed inline in Block A.


In [36]:
# ── teardown sweep: reclaim stragglers (this run + any prior-run leftovers) ───
# Every probe tears down its own catalog + scaff + worker. This belt-and-suspenders
# pass removes anything left behind — including principals from the OLD '-role-'
# naming and any test catalog whose teardown was interrupted — so no residue can
# affect other tests.
TEST_PREFIXES = ["hzr-", "tenantA-", "tenantB-"]

# 1) leftover test catalogs → recursive teardown (drops ns/tables/views + purge)
_cats = requests.get(f"{BASE_MGMT}/catalogs", headers=h(tok))
_names = [(c.get("name") if isinstance(c, dict) else c)
          for c in (_cats.json().get("catalogs", []) if _cats.status_code == 200 else [])]
_left = [n for n in _names if n and any(n.startswith(p) for p in TEST_PREFIXES)]
for n in _left:
    try: teardown_catalog(n, root=tok)
    except Exception as e: print(f"  ⚠️ teardown_catalog({n}): {e}")

# 2) leftover principals / principal-roles (covers current 'worker'/'scaff' AND
#    the deprecated 'role' suffix from earlier runs)
_swept = sweep_instance_principals(
    prefixes=TEST_PREFIXES,
    suffixes=("scaff", "worker", "role", "inst", "full", "scaffold"))

print(f"🧹 cleanup: tore down {len(_left)} leftover test catalog(s); "
      f"swept {_swept} principal/role object(s). Polaris is clean.")

🧹 swept 0 leftover instance principal/role objects
🧹 cleanup: tore down 0 leftover test catalog(s); swept 0 principal/role object(s). Polaris is clean.
